In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import copy
import pandas as pd

In [3]:
# Base Mandats

In [10]:
here = os.getcwd()
global_dir = os.path.dirname(here)
data_dir = os.path.join(global_dir, "0_raw\\json\\acteur")

In [11]:
# DEBUG
places = []
for filename in os.listdir(data_dir):
    with open(os.path.join(data_dir, filename), "r", encoding="UTF-8") as input_file:
        content = json.load(input_file)

    mandats = content["acteur"]["mandats"]["mandat"]
    for mandat in mandats:
      if type(mandat) == dict:
        if "mandature" in mandat:
          if type(mandat.get("mandature")) == dict:
            if "placeHemicycle" in mandat.get("mandature"):
              places.append(mandat.get("mandature").get("placeHemicycle"))

In [12]:
len(list(set(places)))

576

In [14]:
# Test
test = os.path.join(data_dir, "PA226.json")

# Carica il JSON
with open(test, "r", encoding="utf-8") as f:
    content = json.load(f)

# Stampa
import pprint
pprint.pprint(content, width=120)

{'acteur': {'@xmlns': 'http://schemas.assemblee-nationale.fr/referentiel',
            'adresses': {'adresse': [{'@xmlns:xsi': 'http://www.w3.org/2001/XMLSchema-instance',
                                      '@xsi:type': 'AdresseSiteWeb_Type',
                                      'adresseDeRattachement': None,
                                      'poids': None,
                                      'type': '22',
                                      'typeLibelle': 'Site internet',
                                      'uid': 'AD293672',
                                      'valElec': 'www.jpabelin.com'},
                                     {'@xmlns:xsi': 'http://www.w3.org/2001/XMLSchema-instance',
                                      '@xsi:type': 'AdressePostale_Type',
                                      'adresseDeRattachement': None,
                                      'codePostal': '75355',
                                      'complementAdresse': None,
                 

In [16]:
## Check to see which json files do not have an address
files = [f for f in os.listdir(data_dir) if f.endswith(".json")]

count_with_adresse = 0
count_without_adresse = 0
count_errors = 0

for filename in files:
    filepath = os.path.join(data_dir, filename)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = json.load(f)
        
        acteur = content.get("acteur", None)
        if acteur is None:
            count_without_adresse += 1
            continue
        
        adresses = acteur.get("adresses", None)
        if adresses is None:
            count_without_adresse += 1
            continue
        
        adresse = adresses.get("adresse", None)
        if adresse is None:
            count_without_adresse += 1
        else:
            count_with_adresse += 1
    except Exception as e:
        count_errors += 1

print(f"Files with 'adresse': {count_with_adresse}")
print(f"Files without 'adresse': {count_without_adresse}")
print(f"Files with errors: {count_errors}")

Files with 'adresse': 2998
Files without 'adresse': 94
Files with errors: 0


In [17]:
database = []

# For each document in the data dir
for filename in os.listdir(data_dir):
    # Final payload that will contain the payload of the actor
    acteur_payload = {}

    # Remove the extension of the file to get the acteurRef
    acteurRef, extension = os.path.splitext(filename)

    # Open the file and load the json payload
    with open(os.path.join(data_dir, filename), "r", encoding="UTF-8") as input_file:
        content = json.load(input_file)

    # Fetchin basic etat civil
    etat_civil = {k: v for k, v in content.get("acteur").get("etatCivil").get("ident").items() if k in ["civ", "prenom", "nom", "alpha"]}
    etat_civil.update({"acteurRef": acteurRef})

    date_naissance = content.get("acteur").get("etatCivil").get("infoNaissance").get("dateNais")
    if type(date_naissance) != dict:
        etat_civil.update({"dateNais": date_naissance})

    ville_naissance = content.get("acteur").get("etatCivil").get("infoNaissance").get("villeNais")
    if type(ville_naissance) != dict:
        etat_civil.update({"villeNais": ville_naissance})

    dep_naissance = content.get("acteur").get("etatCivil").get("infoNaissance").get("depNais")
    if type(dep_naissance) != dict:
        etat_civil.update({"depNais": dep_naissance})

    pays_naissance = content.get("acteur").get("etatCivil").get("infoNaissance").get("paysNais")
    if type(pays_naissance) != dict:
        etat_civil.update({"paysNais": pays_naissance})

    date_deces = content.get("acteur").get("etatCivil").get("dateDeces")
    if type(date_deces) != dict:
        etat_civil.update({"dateDeces": date_deces})

    acteur_payload.update(etat_civil)

    # ==========================================================
    permanence_adresse = dict()

    # Controllo se 'adresse' esiste dentro content["acteur"]["adresses"]
    if "adresse" in content["acteur"].get("adresses", {}):
        adresses = content["acteur"]["adresses"]["adresse"]
    if type(adresses) == dict:
        adresses = [adresses]

    for adresse in adresses:
        if adresse["type"] == 2:
            permanence_adresse["permanence_codePostal"] = adresse.get("codePostal")
            permanence_adresse["permanence_ville"] = adresse.get("ville")

    # Se non esiste 'adresse', permanence_adresse rimane vuoto
    acteur_payload.update(permanence_adresse)

    # ==========================================================

    profession = {
        "profession_libelleCourant": content["acteur"]["profession"]["libelleCourant"],
        "profession_catSocPro": content["acteur"]["profession"]["socProcINSEE"]["catSocPro"],
        "profession_famSocPro": content["acteur"]["profession"]["socProcINSEE"]["famSocPro"],
    }

    for k, v in profession.items():
        if type(v) == dict:
            profession[k] = None

    acteur_payload.update(profession)

    # ============================================================

    formatted_mandats = []

    mandats = content["acteur"]["mandats"]["mandat"]

    if type(mandats) == dict:
        mandats = [mandats]

    for mandat in mandats:
        payload = {
            "mandat_type": mandat.get("@xsi:type"),
            "mandat_uid": mandat.get("uid"),
            "mandat_organe": mandat.get("organes").get("organeRef"),
            "mandat_type_organe": mandat.get("typeOrgane"),
            "mandat_debut": mandat.get("dateDebut"),
            "mandat_fin": mandat.get("dateFin"),
            "mandat_qualite": mandat.get("infosQualite").get("codeQualite"),
            "mandat_legislature": mandat.get("legislature"),
        }
        if payload["mandat_type"] == "MandatParlementaire_type":
            payload.update(
                {"mandat_election_lieu_{}".format(k): v for k, v in mandat.get("election").get("lieu").items()}
            )
            payload.update({
                "mandat_mandature_{}".format(k):v for k, v in mandat.get("mandature").items()
            })

        formatted_mandats.append(payload)

    for mandat in formatted_mandats:

        acteur_copy = copy.deepcopy(acteur_payload)
        acteur_copy.update(mandat)
        database.append(acteur_copy)

In [18]:
df = pd.DataFrame.from_records(database)

In [28]:
df.sort_values(by=["mandat_legislature", "mandat_debut"], ascending=[False, False], inplace=True)
df.head(5)

,civ,prenom,nom,alpha,acteurRef,dateNais,villeNais,depNais,paysNais,profession_libelleCourant,...,mandat_election_lieu_regionType,mandat_election_lieu_departement,mandat_election_lieu_numDepartement,mandat_election_lieu_numCirco,mandat_mandature_datePriseFonction,mandat_mandature_causeFin,mandat_mandature_premiereElection,mandat_mandature_placeHemicycle,mandat_mandature_mandatRemplaceRef,dateDeces
102901,M.,Didier,Le Gac,Le Gac,PA719404,1965-07-19,Castres,Tarn,France,Profession rattachée à l'enseignement,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104360,Mme,Liliana,Tanguy,Tanguy,PA719550,1967-03-12,Pancevo,NaN,Serbie,Cadre supérieur du secteur public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115719,M.,Bastien,Lachaud,Lachaud,PA720846,1980-08-05,Vitry sur Seine,Val-de-Marne,France,Professeur du secondaire et technique,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
116123,Mme,Mathilde,Panot,Panot,PA720892,1989-01-15,Tours,Indre-et-Loire,France,Coordinatrice de projets associatifs,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154365,M.,Pierre,Cazeneuve,Cazeneuvep,PA795864,1995-03-11,Saint Cloud,Hauts-de-Seine,France,Cadre de la fonction publique,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
df.to_csv("mandats.csv")

In [30]:
df[(df["mandat_election_lieu_numCirco"] == "2") & (df["mandat_type"] == "MandatParlementaire_type") & (df["mandat_election_lieu_numDepartement"] == "974")]

,civ,prenom,nom,alpha,acteurRef,dateNais,villeNais,depNais,paysNais,profession_libelleCourant,...,mandat_election_lieu_regionType,mandat_election_lieu_departement,mandat_election_lieu_numDepartement,mandat_election_lieu_numCirco,mandat_mandature_datePriseFonction,mandat_mandature_causeFin,mandat_mandature_premiereElection,mandat_mandature_placeHemicycle,mandat_mandature_mandatRemplaceRef,dateDeces
134768,Mme,Karine,Lebon,Lebon,PA774962,1985-06-09,Le Port,Réunion,France,Professeure des écoles,...,Dom,Réunion,974,2,2024-07-08,None,1,576,None,NaN
134756,Mme,Karine,Lebon,Lebon,PA774962,1985-06-09,Le Port,Réunion,France,Professeure des écoles,...,Dom,Réunion,974,2,2022-06-22,Fin de législature,1,576,None,NaN
134767,Mme,Karine,Lebon,Lebon,PA774962,1985-06-09,Le Port,Réunion,France,Professeure des écoles,...,Dom,Réunion,974,2,2020-09-28,Fin de législature,0,576,None,NaN
74065,Mme,Huguette,Bello,Bello,PA441,1950-08-24,Ravine-des-Cabris,Réunion,France,Directrice d'école maternelle,...,Dom,Réunion,974,2,2017-06-21,Démission,1,None,None,NaN
74066,Mme,Huguette,Bello,Bello,PA441,1950-08-24,Ravine-des-Cabris,Réunion,France,Directrice d'école maternelle,...,Dom,Réunion,974,2,2012-06-20,Fin de législature,1,None,None,NaN
74067,Mme,Huguette,Bello,Bello,PA441,1950-08-24,Ravine-des-Cabris,Réunion,France,Directrice d'école maternelle,...,Dom,Réunion,974,2,2007-06-20,Fin de législature,1,None,None,NaN
74064,Mme,Huguette,Bello,Bello,PA441,1950-08-24,Ravine-des-Cabris,Réunion,France,Directrice d'école maternelle,...,Dom,Réunion,974,2,2002-06-19,Fin de législature,1,None,None,NaN


In [31]:
df["mandat_mandature_placeHemicycle"].nunique()

575

In [33]:
df[(df["mandat_type"] == "MandatParlementaire_type") & (df["mandat_type_organe"] == "ASSEMBLEE") & (df["mandat_fin"].isna())]

,civ,prenom,nom,alpha,acteurRef,dateNais,villeNais,depNais,paysNais,profession_libelleCourant,...,mandat_election_lieu_regionType,mandat_election_lieu_departement,mandat_election_lieu_numDepartement,mandat_election_lieu_numCirco,mandat_mandature_datePriseFonction,mandat_mandature_causeFin,mandat_mandature_premiereElection,mandat_mandature_placeHemicycle,mandat_mandature_mandatRemplaceRef,dateDeces
151686,Mme,Nathalie,Coggia,Coggia,PA795402,1977-07-16,France,Yvelines,France,Directrice financière et des opérations,...,Français établis hors de France,Français établis hors de France,099,5,2025-10-13,None,0,326,None,NaN
162448,M.,Pierre-Henri,Carbonnel,Carbonnel,PA841973,1990-03-23,Montauban,Tarn-et-Garonne,France,agriculteur,...,Métropolitain,Tarn-et-Garonne,82,1,2025-10-13,None,0,098,None,NaN
69294,M.,Michel,Barnier,Barnier,PA368,1951-01-09,La Tronche,Isère,France,Ancien Premier ministre,...,Métropolitain,Paris,75,2,2025-09-29,None,1,180,None,NaN
163615,M.,Lionel,Duparay,Duparay,PA870009,1981-02-07,Châtenay-Malabry,Hauts-de-Seine,France,Ingénieur,...,Métropolitain,Saône-et-Loire,71,5,2025-11-13,None,0,084,None,NaN
59043,Mme,Marie-Christine,Dalloz,Dalloz,PA332523,1958-01-10,Saint-Claude,Jura,France,retraitée,...,Métropolitain,Jura,39,2,2025-04-07,None,1,168,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158177,M.,Alexandre,Dufosset,Dufosset,PA840135,1998-08-24,Maubeuge,Nord,France,(54) - Employé administratif d'entreprise,...,Métropolitain,Nord,59,18,2024-07-01,None,0,038,None,NaN
158217,M.,Guillaume,Florquin,Florquin,PA840143,1993-05-04,Roncq,Nord,France,(33) - Cadre de la fonction publique,...,Métropolitain,Nord,59,20,2024-07-01,None,0,130,None,NaN
158245,M.,Bruno,Clavet,Clavet,PA840159,1989-05-06,Aubagne,Bouches-du-Rhône,France,(45) - Profession intermédiaire administrative...,...,Métropolitain,Pas-de-Calais,62,3,2024-07-01,None,0,043,None,NaN
158309,M.,Emmanuel,Grégoire,Gregoire,PA840171,1977-12-24,Les Lilas,Seine-Saint-Denis,France,(37) - Cadre administratif et commercial d'ent...,...,Métropolitain,Paris,75,7,2024-07-01,None,0,419,None,NaN


In [34]:
# datePriseFonction (2024-07-08)

In [35]:
df[(df["mandat_type"] == "MandatParlementaire_type") & (df["mandat_type_organe"] == "ASSEMBLEE") & (df["mandat_mandature_datePriseFonction"] == "2024-07-08")]

,civ,prenom,nom,alpha,acteurRef,dateNais,villeNais,depNais,paysNais,profession_libelleCourant,...,mandat_election_lieu_regionType,mandat_election_lieu_departement,mandat_election_lieu_numDepartement,mandat_election_lieu_numCirco,mandat_mandature_datePriseFonction,mandat_mandature_causeFin,mandat_mandature_premiereElection,mandat_mandature_placeHemicycle,mandat_mandature_mandatRemplaceRef,dateDeces
144,M.,Alain,David,DavidA,PA1008,1949-06-02,Libourne,Gironde,France,Ingénieur,...,Métropolitain,Gironde,33,4,2024-07-08,None,1,410,None,NaN
5067,M.,Nicolas,Forissier,Forissier,PA1327,1961-02-17,Paris,Paris,France,Chef d'entreprise,...,Métropolitain,Indre,36,2,2024-07-08,Nomination comme membre du Gouvernement,1,None,None,NaN
8169,M.,Jérôme,Guedj,Guedj,PA1567,1972-01-23,Pantin,Seine-Saint-Denis,France,Inspecteur général des affaires sociales,...,Métropolitain,Essonne,91,6,2024-07-08,None,1,417,None,NaN
8913,M.,David,Habib,HabibD,PA1592,1961-03-16,Paris,Paris,France,Cadre,...,Métropolitain,Pyrénées-Atlantiques,64,3,2024-07-08,None,1,455,None,NaN
9617,M.,Michel,Herbillon,Herbillon,PA1630,1951-03-06,Saint-Mandé,Paris,France,Cadre supérieur,...,Métropolitain,Val-de-Marne,94,8,2024-07-08,None,1,175,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163216,M.,Emmanuel,Maurel,Maurel,PA842271,1973-05-10,Epinay-sur-Seine,Seine-Saint-Denis,France,"(34) - Professeur, profession scientifique",...,Métropolitain,Val-d'Oise,95,3,2024-07-08,None,0,584,None,NaN
163244,Mme,Anchya,Bamana,Bamana,PA842279,1971-03-12,Sada,Mayotte,France,(33) - Cadre de la fonction publique,...,Dom,Mayotte,976,2,2024-07-08,None,0,026,None,NaN
163274,M.,Emmanuel,Tjibaou,Tjibaou,PA842299,1976-03-17,Nouméa,Nouvelle-Calédonie,France,(33) - Cadre de la fonction publique,...,Collectivités d'outre-mer et Nouvelle-Calédonie,Nouvelle-Calédonie,988,2,2024-07-08,None,0,582,None,NaN
163335,M.,Vincent,Caure,Caure,PA842311,1992-12-31,Castres,Tarn,France,(33) - Cadre de la fonction publique,...,Français établis hors de France,Français établis hors de France,099,3,2024-07-08,None,0,384,None,NaN


In [36]:
# Base Organe

In [39]:
organe_dir = os.path.join(global_dir, "0_raw\\json\\organe")

In [41]:
database2 = []

# For each document in the data dir
for filename in os.listdir(organe_dir):

    # Remove the extension of the file to get the organeRef
    organeRef, extension = os.path.splitext(filename)

    # Open the file and load the json payload
    with open(os.path.join(organe_dir, filename), "r", encoding="UTF-8") as input_file:
        content = json.load(input_file)

    # Fetchin variables
    payload = {
      "uid": content.get("organe").get("uid"),
      "codeType": content.get("organe").get("codeType"),
      "libelleAbrege": content.get("organe").get("libelleAbrege"),
      "libelle": content.get("organe").get("libelle"),
    }

    database2.append(payload)

In [42]:
df2 = pd.DataFrame.from_records(database2)

In [43]:
df2

,uid,codeType,libelleAbrege,libelle
0,PO191887,ORGEXTPARL,Mines antipersonnel,Commission nationale pour l'élimination des mi...
1,PO201115,API,APF,Section française de l'Assemblée parlementaire...
2,PO201269,DELEG,Délégation de l'Assemblée nationale aux droits...,Délégation de l'Assemblée nationale aux droits...
3,PO201275,DELEG,Délégation de l'Assemblée nationale à l'aménag...,Délégation de l'Assemblée nationale à l'aménag...
4,PO201361,ORGEXTPARL,Laboratoire de Bure,Comité local d'information et de suivi du labo...
...,...,...,...,...
10747,PO874468,MISINFO,Evolution du pouvoir d'achat en France depuis ...,Mission d’information sur l’évolution du pouvo...
10748,PO874480,CNPE,CE Audiovisuel public,"Commission d’enquête sur la neutralité, le fon..."
10749,PO874906,MISINFO,comment l’intelligence artificielle transforme...,"Mission d’information sur « création, diffusio..."
10750,PO875512,CMP,PLFSS 2026,Commission mixte paritaire chargée de proposer...


In [44]:
df2.to_csv("organes.csv")